✅ Typical cleaning workflow in PySpark

Remove unwanted characters → regexp_replace()
Convert to date → to_date()
Handle multiple formats → coalesce()
Standardize format → date_format()
current_date()
date_add()
date_sub()
datediff()
add_months()
to_date()


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = SparkSession.builder.getOrCreate()

# Sample data with messy date formats
data = [
    (1, "2026-03-01"),
    (2, "03/02/2026"),
    (3, "March 3, 2026"),
    (4, "03-04-26"),      # ambiguous short year
    (5, "2026.03.05"),
    (6, "05-Mar-2026"),
    (7, "20260306"),      # compact format
    (8, "07/03/26"),      # DD/MM/YY vs MM/DD/YY ambiguity
    (9, "2026/03/08"),
    (10, "09th Mar 2026"),
    (11,None)
]

columns = ["id","date"]
df = spark.createDataFrame(data, columns)
df.display()


In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import (
    col, regexp_replace, to_date, coalesce, lit, date_format, trim
)

df2 = df.withColumn(
    "date_clean",
    trim(
        regexp_replace(
            regexp_replace(
                regexp_replace(col("date"), "(?i)(st|nd|rd|th)", ""),  
                "[./-]", "/"                                           
            ),
            "\\s+", " "                                                
        )
    )
).withColumn(
    "date_parsed",
    coalesce(
        to_date(col("date_clean"), "yyyy/MM/dd"),
        to_date(col("date_clean"), "dd/MM/yyyy"),
        to_date(col("date_clean"), "MM/dd/yyyy"),
        to_date(col("date_clean"), "dd/MM/yy"),
        to_date(col("date_clean"), "MM/dd/yy"),
        to_date(col("date_clean"), "MMMM d, yyyy"),
        to_date(col("date_clean"), "d MMM yyyy"),
        to_date(col("date_clean"), "dd MMM yyyy"),
        to_date(col("date_clean"), "yyyyMMdd"),
        to_date(lit("1900-01-01"))   
    )
).withColumn(
    "date_final",
    date_format(col("date_parsed"), "yyyy-MM-dd")
)

df2.display()

In [0]:
from pyspark.sql.functions import expr

df_clean2 = df.withColumn("clean_date",
    expr("""
        coalesce(
            try_to_date(date, 'yyyy-MM-dd'),
            try_to_date(date, 'MM/dd/yyyy'),
            try_to_date(date, 'dd-MMM-yyyy'),
            try_to_date(date, 'yyyy.MM.dd'),
            try_to_date(date, 'yyyyMMdd'),
            try_to_date(date, 'dd/MM/yy'),
            try_to_date(date, 'MM-dd-yy'),
            try_to_date(date, 'MMMM d, yyyy'),
            try_to_date(date, 'yyyy/MM/dd'),
         
        )
    """)
)

df_clean2.display()

In [0]:
df = df.withColumn("date", to_date(col('date'), "dd-MM-yyyy"))


In [0]:
from pyspark.sql.functions import col, to_date, unix_timestamp, coalesce, expr

# Try multiple formats using coalesce
df1 = df.withColumn("to_date", coalesce(to_date(col("date"), "yyyy-MM-dd"), to_date(col("date"), "MM/dd/yyyy"))).filter(col("id")<> 2)
df1.display()

In [0]:
from pyspark.sql.functions import expr
df2 = df.withColumn("Try_do_date1",expr("try_to_date(raw_date, 'yyyy-MM-dd')")) \
        .withColumn("Try_do_date2",expr("try_to_date(raw_date, 'MM-dd-yyyy')")) \
        .withColumn("Try_do_date3",expr("try_to_date(raw_date, 'dd-MM-yyyy')"))

df2.display()